# 02. Machine Learning Pipeline, Training & Serialization

**Project:** AttriGuard - Employee Attrition Prediction System  
**Objective:** Build a robust, leak-free scikit-learn pipeline, train multiple candidate models (Logistic Regression, Random Forest, Gradient Boosting), tune hyperparameters, evaluate recall on minority class, and serialize the champion pipeline for FastAPI deployment.

In [1]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, 
    roc_curve, precision_recall_curve, f1_score, recall_score, precision_score
)

## 1. Load Data & Clean

In [2]:
df = pd.read_csv('../data/WA_Fn-UseC_-HR-Employee-Attrition.csv')

# Drop uninformative constants and surrogate keys
drop_cols = ['EmployeeCount', 'StandardHours', 'Over18', 'EmployeeNumber']
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

# Encode target: Yes -> 1, No -> 0
df['Attrition'] = df['Attrition'].map({'Yes': 1, 'No': 0})

X = df.drop(columns=['Attrition'])
y = df['Attrition']

print(f"Feature matrix: {X.shape}, Target: {y.shape} (Attrition rate: {y.mean():.2%})")

## 2. Stratified Train / Test Split (Preventing Data Leakage)

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples, Holdout test set: {X_test.shape[0]} samples")

## 3. Construct Preprocessing Pipeline

In [4]:
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', num_transformer, num_cols),
    ('cat', cat_transformer, cat_cols)
])

print(f"Preprocessed features count: {len(num_cols)} numerical, {len(cat_cols)} categorical")

## 4. Candidate Model Benchmarking & Metric Comparison

In [5]:
models = {
    'Logistic Regression (Balanced)': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Random Forest (Balanced)': RandomForestClassifier(n_estimators=150, max_depth=8, class_weight='balanced', random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=120, learning_rate=0.08, max_depth=4, random_state=42)
}

benchmark_records = []
fitted_pipelines = {}

for name, clf in models.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', clf)
    ])
    pipe.fit(X_train, y_train)
    fitted_pipelines[name] = pipe
    
    y_prob = pipe.predict_proba(X_test)[:, 1]
    y_pred = pipe.predict(X_test)
    
    benchmark_records.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision (Yes)': precision_score(y_test, y_pred, zero_division=0),
        'Recall (Yes)': recall_score(y_test, y_pred, zero_division=0),
        'F1-Score': f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, y_prob)
    })

results_df = pd.DataFrame(benchmark_records)
results_df

## 5. ROC Curve & Precision-Recall Curve Comparison

In [6]:
plt.figure(figsize=(14, 5))

# ROC Curve
plt.subplot(1, 2, 1)
for name, pipe in fitted_pipelines.items():
    y_prob = pipe.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], 'k--', label='Random Chance')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curves')
plt.legend()

# PR Curve
plt.subplot(1, 2, 2)
for name, pipe in fitted_pipelines.items():
    y_prob = pipe.predict_proba(X_test)[:, 1]
    prec, rec, _ = precision_recall_curve(y_test, y_prob)
    plt.plot(rec, prec, label=name)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curves (Class 1)')
plt.legend()

plt.tight_layout()
plt.show()

## 6. Champion Model Selection & Serialization

In [7]:
champion_name = 'Gradient Boosting'
champion_pipeline = fitted_pipelines[champion_name]

os.makedirs('../models', exist_ok=True)
model_path = '../models/attrition_pipeline.pkl'
joblib.dump(champion_pipeline, model_path)
print(f"Successfully serialized champion pipeline to: {model_path}")